# 07 — Three-Engine Evidence Tree & Explain Demo

One schema, three engines, shared Store. All output from real framework calls.

**Demonstrates:**
1. **Evidence tree** — full node hierarchy with multi-rule chains (recursive_depth > 0)
2. **Certainty** — condition_weights + confidence → bottleneck vs additive
3. **ProbLog proof tree** — deep proof_goal/proof_leaf hierarchy + probability pipeline
4. **PyReason timeline** — multi-node graph propagation across timesteps, multiple chains
5. **Cross-engine data flow** — accepted facts from one engine feed the next
6. **Full explain pipeline** — tree/timeline → summary → narrative → NL → HTML for each engine

ProbLog and PyReason use mocked runners; the entire evaluate → accept → explain pipeline is real.

## 0. Imports

In [ ]:
import sys, tempfile
from pathlib import Path
from pprint import pprint
from unittest.mock import patch

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field, Relationship, Rule, Pred, vars as sdk_vars
from factpy_kernel.sdk.compile import compile_schema_from_classes
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation, accept_runtime_derivation,
    explain_runtime_tree, explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    explain_runtime_timeline, explain_runtime_timeline_summary, explain_runtime_timeline_narrative,
)
from factpy_kernel.audit.evidence_graph import render_evidence_graph_html
from factpy_kernel.audit.static_ui import render_candidate_evidence_html
import factpy_kernel.adapters.problog
import factpy_kernel.adapters.pyreason
from factpy_kernel.adapters.pyreason.session import PyReasonSession
from factpy_kernel.adapters.pyreason.runner import PyReasonRunConfig, PyReasonRunResult
from IPython.display import HTML, display

In [ ]:
def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    parts = [kind]
    if kind == "candidate_result":
        parts.append(f"root_result_kind={node.get('root_result_kind', '?')}")
        em = node.get("engine_meta")
        if em: parts.append(f"engine_meta={em}")
    elif kind == "predicate_witness_group":
        parts.append(f"pred_id={node.get('pred_id', '?')}")
        cc = node.get("condition_confidence")
        if cc is not None: parts.append(f"cc={cc}")
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        parts.append(f"[{', '.join(c.get('val','?') for c in claims)}]")
        conf = node.get("confidence")
        if conf is not None: parts.append(f"conf={conf}")
    elif kind in ("proof_goal", "proof_leaf"):
        parts.append(f"pred={node.get('pred_id', '?')}")
        args = node.get("goal_args", [])
        if args: parts.append(f"args={args[:2]}{'...' if len(args)>2 else ''}")
    elif kind == "rule_ref":
        parts.append(f"{node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}")
    elif kind == "non_fact_check":
        parts.append(f"{node.get('check_kind', '?')} status={node.get('status', '?')}")
    print(f"{prefix}- {' | '.join(parts)}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

## 1. Schema + Session

Research collaboration network with Relationship edges.

In [ ]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    h_index: str = Field(cardinality="single")
    publications: str = Field(cardinality="single")
    is_established: str = Field(cardinality="single")
    qualifies_grant: str = Field(cardinality="single")
    tag_seed: str = Field(cardinality="single")
    tag: str = Field(cardinality="single")
    risk_flag: str = Field(cardinality="single")

class Collaboration(Relationship):
    from_entity = Researcher
    to_entity = Researcher
    joint_papers: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Researcher, Collaboration])
sdk = SDKStore([Researcher], schema_ir=schema_ir)

# ── Multi-rule chain: established_check → grant_qualification ──
with sdk_vars("r", "exp", "h") as (r, exp, h):
    established_check = Rule(
        id="q.established_check", version="1.0.0",
        select=[r, exp],
        where=[Pred("researcher:expertise", r, exp),
               Pred("researcher:h_index", r, h),
               Pred("researcher:publications", r, "50+")],
        expose=True,
        condition_weights={"b0.a0": 0.7, "b0.a1": 0.9, "b0.a2": 0.5},
    )

with sdk_vars("r", "exp", "pub") as (r, exp, pub):
    grant_rule = Rule(
        id="q.grant_qualification", version="1.0.0",
        select=[r, exp],
        where=[RuleRef(established_check)(r, exp),
               Pred("researcher:publications", r, pub)],
        expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.6},
    )

# Registry + session
registry_dir = tempfile.mkdtemp(prefix="three_engine_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
for rule in [established_check, grant_rule]:
    registry.register_rule_spec(sdk._compile_rule_input(rule))

reset_runtime_sessions_for_tests()
resp = open_runtime_session({"registry_root": registry_dir})
session_id = resp["session"]["session_id"]

# Seed 3 researchers
alice_ref = sdk.ref(Researcher, researcher_id="Alice")
bob_ref = sdk.ref(Researcher, researcher_id="Bob")
carol_ref = sdk.ref(Researcher, researcher_id="Carol")

researchers = [
    (alice_ref, "Alice Chen",   "NLP",      "42", "50+", 0.95, 0.88, 0.7),
    (bob_ref,   "Bob Zhang",    "CV",       "35", "50+", 0.90, 0.75, 0.6),
    (carol_ref, "Carol Li",     "RL",       "28", "30",  0.85, 0.65, 0.5),
]
for ref, name, exp, h_idx, pubs, exp_conf, h_conf, pub_conf in researchers:
    write_runtime_fact(session_id, {"pred_id": "researcher:name", "e_ref": ref,
        "rest_terms": [["string", name]]}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:expertise", "e_ref": ref,
        "rest_terms": [["string", exp]], "meta": {"confidence": exp_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:h_index", "e_ref": ref,
        "rest_terms": [["string", h_idx]], "meta": {"confidence": h_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:publications", "e_ref": ref,
        "rest_terms": [["string", pubs]], "meta": {"confidence": pub_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:tag_seed", "e_ref": ref,
        "rest_terms": [["string", "senior"]], "meta": {"confidence": 1.0}}, kind="add")

print("Session ready. 3 researchers seeded:")
print("  Alice: NLP  h=42 pubs=50+ (exp=0.95, h=0.88, pub=0.7)")
print("  Bob:   CV   h=35 pubs=50+ (exp=0.90, h=0.75, pub=0.6)")
print("  Carol: RL   h=28 pubs=30  (exp=0.85, h=0.65, pub=0.5)")

---
## 2. Native Engine — Multi-Rule Evidence Tree with Certainty

`grant_qualification` chains through `established_check` via `RuleRef` → creates
a tree with `referenced_support` subtree and `recursive_depth > 0`.

### 2.1 Evaluate + Accept

In [ ]:
eval_native = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.grant", "version": "1.0.0",
    "target": "researcher:qualifies_grant", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.grant_qualification", "1.0.0", ["$r", "$exp"]]],
    "mode": "native",
}})

native_cands = eval_native["evaluation"]["candidates"]
for c in native_cands:
    accept_runtime_derivation(session_id, {"candidate": c})

cid_native = native_cands[0]["candidate_id"]
print(f"Candidates: {len(native_cands)} (Alice & Bob qualify, Carol has pubs=30)")
print(f"  confidence_kind: {native_cands[0]['confidence_kind']} ← auto-routed")

### 2.2 Evidence Tree — Multi-Rule Node Hierarchy

Shows the full chain: `candidate_result` → `rule_ref(grant_qualification)` →
`referenced_support` → `rule_ref(established_check)` → witness groups + assertions.

`recursive_depth` will be > 0 due to the chained `RuleRef`.

In [ ]:
tree_resp = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_native})

print("=== Evidence Tree (Native — Multi-Rule Chain) ===")
print_tree(tree_resp["tree"]["root"])

### 2.3 Certainty — Bottleneck vs Additive

In [ ]:
s_bn = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
s_ad = explain_runtime_summary(session_id, {
    "kind": "candidate", "id": cid_native, "certainty_aggregation": "additive"})
cs_bn, cs_ad = s_bn.get("certainty_summary"), s_ad.get("certainty_summary")

print(f"recursive_depth = {s_bn['summary']['recursive_depth']}")
print(f"witness_assertion_count = {s_bn['summary']['witness_assertion_count']}")
print(f"rule_ref_count = {s_bn['summary']['rule_ref_count']}")

if cs_bn and cs_ad:
    print(f"\n{'':25s} {'Bottleneck':>12s}  {'Additive':>12s}")
    print(f"{'Aggregate':25s} {cs_bn['aggregate_certainty']:>12}  {cs_ad['aggregate_certainty']:>12}")
    for b, a in zip(cs_bn["conditions"], cs_ad["conditions"]):
        print(f"  {b['atom_key']:23s} {b['impact']:>12}  {a['impact']:>12}")

### 2.4 Narrative + NL

In [ ]:
narr = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_native})
nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_native})

n = narr["narrative"]
print(f"Headline: {n['headline']}")
for section in ["overview_lines", "evidence_lines", "rule_chain_lines", "certainty_lines"]:
    lines = n.get(section, [])
    if lines:
        print(f"\n{section}:")
        for line in lines: print(f"  {line}")
bn = n.get("certainty_bottleneck")
if bn: print(f"  bottleneck: {bn}")

print(f"\nNL ({len(nl['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:150]}{'...' if len(p)>150 else ''}")

### 2.5 HTML Rendering

In [ ]:
html_native = render_candidate_evidence_html(tree_resp["tree"], narrative=narr.get("narrative"))
print(f"HTML: {len(html_native)} chars")
display(HTML(f"<div style='border:2px solid #3498db;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#3498db;color:white;padding:6px 12px;font-size:13px'>"
             f"Native — Multi-Rule Evidence Tree + Certainty</div>"
             f"<iframe srcdoc=\'{html_native.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:500px;border:none'></iframe></div>"))

---
## 3. ProbLog — Deep Proof Tree with Probability

Mocked with a 3-level proof trace: `answer` → `tag_seed` → `name` + `expertise`.
Shows `proof_goal` with children (recursive proof steps) + multiple `proof_leaf` nodes.

In [ ]:
def _mock_problog_deep():
    a = sdk.ref(Researcher, researcher_id="Alice")
    return "\n".join([
        " call query(X1,X2) {0.00000} []",
        f'  result query(X1,X2) ("senior","{a}") {{{{}}}} {{0.00012}} []',
        " complete query(X1,X2) {0.00013} {0.00013} []",
        f' call answer("senior","{a}") {{0.00019}} [at 4:7]',
        # Level 1: tag_seed goal (has children → proof_goal)
        f'  call researcher__tag_seed("senior","{a}") {{0.00026}} [at 3:9]',
        # Level 2: name goal (leaf → proof_leaf)
        f'   call researcher__name("Alice Chen","{a}") {{0.00032}} [at 2:5]',
        f'    result researcher__name("Alice Chen","{a}") ("Alice Chen","{a}") {{{{}}}} {{0.00040}} [at 2:5]',
        f'   complete researcher__name("Alice Chen","{a}") {{0.00041}} {{0.00009}} []',
        # Level 2: expertise goal (leaf → proof_leaf)
        f'   call researcher__expertise("NLP","{a}") {{0.00045}} [at 2:8]',
        f'    result researcher__expertise("NLP","{a}") ("NLP","{a}") {{{{}}}} {{0.00052}} [at 2:8]',
        f'   complete researcher__expertise("NLP","{a}") {{0.00053}} {{0.00008}} []',
        # Level 1: tag_seed result
        f'   result researcher__tag_seed("senior","{a}") ("senior","{a}") {{{{}}}} {{0.00058}} [at 3:9]',
        f'  complete researcher__tag_seed("senior","{a}") {{0.00059}} {{0.00033}} []',
        # answer result
        f'  result answer("senior","{a}") ("senior","{a}") {{{{}}}} {{0.00065}} []',
        f' complete answer("senior","{a}") {{0.00066}} {{0.00047}} []',
        "",
        f'answer("senior","{a}"):\t0.72',
    ])

with patch("factpy_kernel.adapters.problog.engine_eval.run_problog") as mock_run:
    mock_run.return_value = _mock_problog_deep()
    eval_prob = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.problog_tag", "version": "1.0.0",
        "target": "researcher:tag", "head_vars": ["$u", "$tag"],
        "where": [["pred", "researcher:tag_seed", ["$u", "$tag"]]],
        "mode": "problog",
    }})

prob_cand = eval_prob["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": prob_cand})
cid_prob = prob_cand["candidate_id"]
print(f"ProbLog accepted (support_kind={prob_cand['support_kind']})")

### 3.1 Proof Tree — Deep Hierarchy

`proof_goal` (intermediate, has children) vs `proof_leaf` (terminal, no asrt_id).
Shows 3-level proof: tag_seed ← name + expertise.

In [ ]:
tree_prob = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_prob})

print("=== Evidence Tree (ProbLog — Deep Proof) ===")
print_tree(tree_prob["tree"]["root"])
print(f"\nprobability = {tree_prob['tree']['root']['engine_meta']['probability']}")

### 3.2 Summary + Narrative + NL

In [ ]:
sp = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})["summary"]
np_ = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob})["narrative"]
nl_p = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_prob})

print(f"proof_goal_count = {sp.get('proof_goal_count')}")
print(f"proof_leaf_count = {sp.get('proof_leaf_count')}")
print(f"recursive_depth = {sp.get('recursive_depth')}")
print(f"problog_probability = {sp.get('problog_probability')}")
print(f"\nprobability_lines = {np_.get('probability_lines')}")
print(f"evidence_lines = {np_.get('evidence_lines')}")
print(f"rule_chain_lines = {np_.get('rule_chain_lines')}")

print(f"\nNL ({len(nl_p['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_p["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:150]}{'...' if len(p)>150 else ''}")

### 3.3 HTML Rendering

In [ ]:
html_prob = render_candidate_evidence_html(tree_prob["tree"],
    narrative=explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob}).get("narrative"))
print(f"HTML: {len(html_prob)} chars")
display(HTML(f"<div style='border:2px solid #9b59b6;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#9b59b6;color:white;padding:6px 12px;font-size:13px'>"
             f"ProbLog — Deep Proof Tree + Probability</div>"
             f"<iframe srcdoc=\'{html_prob.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:400px;border:none'></iframe></div>"))

---
## 4. PyReason — Multi-Node Graph Propagation

3 researchers connected: `Alice → Bob → Carol`.
Risk flag propagates across the graph over multiple timesteps.
Shows multiple chains and temporal evolution.

In [ ]:
derived_session = PyReasonSession(schema_ir)
derived_session._write_node_fact_internal("researcher:risk_flag", alice_ref, "true", bound=[1.0, 1.0])
derived_session._write_node_fact_internal("researcher:risk_flag", bob_ref, "true", bound=[0.8, 0.9])

trace_dict = {
    "engine": "pyreason", "trace_type": "event_log", "timesteps": 3,
    "node_events": [
        # Alice: seed at t=0
        {"time": 0, "fixpoint_op": 1,
         "component": alice_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [1.0, 1.0],
         "occurred_due_to": "seed_fact", "clause_groundings": []},
        # Bob: propagated from Alice at t=1
        {"time": 1, "fixpoint_op": 2,
         "component": bob_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [0.8, 0.9],
         "occurred_due_to": "risk_propagation",
         "clause_groundings": [f"[{alice_ref}]", f"[({bob_ref}, {alice_ref})]"]},
        # Carol: propagated from Bob at t=2
        {"time": 2, "fixpoint_op": 3,
         "component": carol_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [0.7, 0.85],
         "occurred_due_to": "risk_propagation",
         "clause_groundings": [f"[{bob_ref}]", f"[({carol_ref}, {bob_ref})]"]},
    ],
    "edge_events": [],
}

with patch("factpy_kernel.adapters.pyreason.engine_eval.run_pyreason") as mock_pr:
    mock_pr.return_value = PyReasonRunResult(
        interpretation=None, trace=None, trace_dict=trace_dict,
        derived_session=derived_session,
        config=PyReasonRunConfig(timesteps=3, atom_trace=True),
        elapsed_seconds=0.02,
    )
    eval_pr = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.pyreason_risk", "version": "1.0.0",
        "target": "researcher:risk_flag", "head_vars": ["$r"],
        "where": [["pred", "researcher:expertise", ["$r", "$exp"]]],
        "mode": "pyreason",
    }})

pr_cand = eval_pr["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": pr_cand})
cid_pr = pr_cand["candidate_id"]
print(f"PyReason accepted. Propagation: Alice(t=0) → Bob(t=1) → Carol(t=2)")

### 4.1 Timeline — Multiple Chains + Temporal Propagation

In [ ]:
tl = explain_runtime_timeline(session_id, {"kind": "candidate", "id": cid_pr})
s_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})
n_pr = explain_runtime_timeline_narrative(session_id, {"kind": "candidate", "id": cid_pr})
nl_pr = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_pr})

timeline = tl["timeline"]
print(f"=== CandidateProvenanceTimeline ===")
print(f"  timesteps: {timeline['timesteps']}, chains: {len(timeline['chains'])}")
for chain in timeline["chains"]:
    comp = chain["component"].split(":")[-1][:20]  # shorten ref
    print(f"\n  Chain: ...{comp}.{chain['label']}" +
          (" [ROOT]" if chain.get("is_root") else ""))
    for evt in chain["events"]:
        print(f"    t={evt['time']}: {evt['old_bound']} → {evt['new_bound']} by {evt['trigger']}")
        if evt.get("groundings"):
            for g in evt["groundings"]: print(f"           grounding: {g[:60]}")

print(f"\nSummary: chains={s_pr['summary']['chain_count']}, "
      f"seeds={s_pr['summary']['seed_count']}, "
      f"propagated={s_pr['summary'].get('propagated_count', '?')}")

print(f"\nNarrative headline: {n_pr['narrative']['headline']}")
for line in n_pr["narrative"].get("propagation_lines", []):
    print(f"  {line[:120]}")

print(f"\nNL ({len(nl_pr['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_pr["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:120]}{'...' if len(p)>120 else ''}")

### 4.2 HTML — Timeline Rendering

In [ ]:
from factpy_kernel.adapters.pyreason.provenance import pyreason_trace_to_evidence_graph, pyreason_trace_from_dict

eg_timeline = pyreason_trace_to_evidence_graph(
    pyreason_trace_from_dict(trace_dict),
    candidate_id=cid_pr,
    candidate_payload=pr_cand.get("payload", {}),
)

html_pr = render_evidence_graph_html(eg_timeline)
print(f"EvidenceGraph: layout={eg_timeline.layout_hint}, nodes={len(eg_timeline.nodes)}, edges={len(eg_timeline.edges)}")
display(HTML(f"<div style='border:2px solid #27ae60;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#27ae60;color:white;padding:6px 12px;font-size:13px'>"
             f"PyReason — Multi-Node Timeline Propagation</div>"
             f"<iframe srcdoc=\'{html_pr.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:350px;border:none'></iframe></div>"))

---
## 5. Cross-Engine Comparison

In [ ]:
print("=" * 70)
print("THREE-ENGINE EXPLAIN COMPARISON")
print("=" * 70)

sn = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
cs = sn.get("certainty_summary")
print(f"\n[Native] CandidateEvidenceTree (multi-rule chain)")
print(f"  witnesses={sn['summary']['witness_assertion_count']}, "
      f"rule_refs={sn['summary']['rule_ref_count']}, "
      f"recursive_depth={sn['summary']['recursive_depth']}")
if cs: print(f"  certainty={cs['aggregate_certainty']} ({cs['aggregation']})")

sp = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})["summary"]
print(f"\n[ProbLog] CandidateEvidenceTree (proof tree)")
print(f"  proof_goals={sp.get('proof_goal_count',0)}, "
      f"proof_leaves={sp.get('proof_leaf_count',0)}, "
      f"depth={sp.get('recursive_depth',0)}")
print(f"  probability={sp.get('problog_probability')}")

st = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})["summary"]
print(f"\n[PyReason] CandidateProvenanceTimeline")
print(f"  chains={st['chain_count']}, seeds={st['seed_count']}, timesteps={st['timesteps']}")

print(f"\n{'=' * 70}")
print("Same schema, same session, same Store.")
print("Three engines, three explain surfaces, one integration bus.")
print(f"{'=' * 70}")

In [ ]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print("Session closed.")

## Architecture Summary

```
  Shared Schema (Researcher + Collaboration) + Store (one session, one ledger)
       │
       ├── Native evaluate → accept → CandidateEvidenceTree
       │     Multi-rule chain: grant_qualification → established_check
       │     predicate_witness_group / assertion_fact + certainty (bottleneck/additive)
       │     recursive_depth > 0, condition_weights flow through RuleRef chains
       │
       ├── ProbLog evaluate → accept → CandidateEvidenceTree
       │     Deep proof tree: proof_goal (intermediate) → proof_leaf (terminal)
       │     problog_probability flows: summary → narrative → NL
       │     proof_leaf has no asrt_id (no dead links in static UI)
       │
       └── PyReason evaluate → accept → CandidateProvenanceTimeline
             Multi-node graph: Alice → Bob → Carol
             Bound propagation across timesteps [1.0,1.0] → [0.8,0.9] → [0.7,0.85]
             explain-timeline endpoints + polymorphic NL dispatch
             EvidenceGraph(timeline) for audit rendering

  Cross-engine: accepted facts from Engine A visible to Engine B
  Unified interface, not unified implementation
```